In [1]:

import xarray as xr
import matplotlib.pyplot as plt
import os
import cartopy
import numpy as np
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import cartopy.crs as ccrs
import matplotlib as mpl
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import pandas as pd
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
import ipywidgets as widgets
import earthkit.data as ekd
import earthkit.plots as ekp
import matplotlib.colors as mcolors
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import io

mpl.rcParams["animation.embed_limit"] = 30

product_parameters = {
    "variable": ["precipitation"],
    "delivery_start": pd.Timestamp("2026-01-01"),
    "delivery_end": pd.Timestamp("2026-06-01"),
    "collection_start": pd.Timestamp("2002-01-01"),
    "collection_end": pd.Timestamp("2026-06-01"),
}

datadir = "../../../datasets/GIRAFE/"
frequencies = ["monthly", "daily"]


### Check that all expected dates are present

In [3]:

# ============================================================
# Check that all expected dates are present for both datasets
# ============================================================

integrity_reports = {}
file_tables = {}

for frequency in frequencies:
    collection_start = product_parameters["collection_start"]
    collection_end = product_parameters["collection_end"]
    temporal_resolution = "MS" if frequency == "monthly" else "D"
    dates = pd.date_range(start=collection_start, end=collection_end, freq=temporal_resolution)

    dir_data = os.path.join(datadir, frequency)
    files = []
    for root, dirs, filenames in os.walk(dir_data):
        for filename in filenames:
            files.append(os.path.join(root, filename))

    df_frequency = pd.DataFrame(files, columns=["file_path"])

    if frequency == "monthly":
        df_frequency["file_date"] = df_frequency["file_path"].str.extract(r"PREmm(\d{6})", expand=False)
        expected_keys = dates.strftime("%Y%m")
    else:
        # GIRAFE daily files use PREdmYYYYMMDD.
        df_frequency["file_date"] = df_frequency["file_path"].str.extract(r"PREdm(\d{8})", expand=False)
        expected_keys = dates.strftime("%Y%m%d")

    file_dates = set(df_frequency["file_date"].dropna())
    dates_set = set(expected_keys)
    missing_dates = sorted(dates_set - file_dates)
    existing_dates = sorted(dates_set & file_dates)

    integrity_reports[frequency] = {
        "frequency": frequency,
        "expected": len(dates_set),
        "existing": len(existing_dates),
        "missing": len(missing_dates),
        "missing_dates": missing_dates,
        "files_found": len(df_frequency),
    }
    file_tables[frequency] = df_frequency.sort_values("file_date").reset_index(drop=True)

    print(f"===== {frequency.upper()} =====")
    print(f"Files found: {len(df_frequency)}")
    if missing_dates:
        print(f"{len(existing_dates)} of {len(dates_set)} expected dates are present.")
        print(f"Missing dates ({len(missing_dates)}):")
        print(missing_dates)
    else:
        print(f"All {len(existing_dates)} expected {frequency} dates are present.")


===== MONTHLY =====
Files found: 294
All 294 expected monthly dates are present.
===== DAILY =====
Files found: 8947
All 8918 expected daily dates are present.


### Check metadata

In [4]:

# ============================================================
# Check metadata for the latest available file in each dataset
# ============================================================

metadata_reports = {}
metadata_tables = {}

for frequency in frequencies:
    df_frequency = file_tables[frequency]
    df_valid = df_frequency.dropna(subset=["file_date"])

    if df_valid.empty:
        print(f"===== {frequency.upper()} METADATA ===== No valid files found.")
        metadata_reports[frequency] = "No valid files found."
        continue

    fname = df_valid.iloc[-1]["file_path"]
    print(f"===== {frequency.upper()} METADATA =====")
    print(fname)

    fieldlist = ekd.from_source("file", fname).to_fieldlist()
    fls = fieldlist.ls()
    print(fls)

    ds_xr = ekd.from_source("file", fname).to_xarray()
    buf = io.StringIO()
    ds_xr.info(buf=buf)
    info_text = buf.getvalue()
    print(info_text)

    metadata_reports[frequency] = {
        "file": fname,
        "fieldlist": fls.to_string(index=False),
        "xarray_info": info_text,
    }
    metadata_tables[frequency] = fls


===== MONTHLY METADATA =====
../../../datasets/GIRAFE/monthly/2026/PREmm20260601000000120IMPGSI1GL.nc
           variable level       valid_datetime units
0     precipitation  None  2026-06-01T00:00:00    mm
1  num_obs_fraction  None  2026-06-01T00:00:00     1
2      num_obs_rate  None  2026-06-01T00:00:00     1
3          num_days  None  2026-06-01T00:00:00     1
4      quality_flag  None  2026-06-01T00:00:00  None
5     num_days_snow  None  2026-06-01T00:00:00     1
xarray.Dataset {
dimensions:
	time = 1 ;
	nv = 2 ;
	lat = 180 ;
	lon = 360 ;

variables:
	datetime64[ns] time(time) ;
		time:long_name = Product dataset time given as seconds since 2000-01-01T00:00:00 ;
		time:standard_name = time ;
		time:axis = T ;
		time:bounds = time_bnds ;
	datetime64[ns] time_bnds(time, nv) ;
		time_bnds:long_name = Time cell boundaries ;
		time_bnds:comment = Contains the start and end times for the time period that the data represent. ;
	float64 lat(lat) ;
		lat:units = degrees_north ;
		lat:long_

In [13]:
# fname = df.loc[df["file_date"] == "202601", "file_path"].iloc[0] # get the file path for a specific date.

# # nc = xr.open_dataset(fname)
# # print(nc.info())
# ds = ekd.from_source("file", fname).to_fieldlist()


# style=ekp.styles.Style(
#     colors="gist_earth_r",
#     levels=[0,0.5,1,2,5,10,20,50,100],
#     ticks=[0,0.5,1,2,5,10,20,50,100],
# )

# chart = ekp.Map()
# # chart.contourf(ds[0],style=style)  
# chart.pcolormesh(ds[0],style=style)

# # Add a title with metadata formatting
# chart.title("GIRAFE monthly averaged {variable_name}\n{time:%B %Y}")

# # Add map decorations
# chart.coastlines()
# chart.borders()
# chart.gridlines()

# # Add a legend
# chart.legend(label="{variable_name} ({units})") 
# chart.show()

### Check values of main fields within the dataset
For monthly datasets, one plot per month per variable is checked.

For daily datasets, at least the file for day 1 of each month shall be checked.

In [5]:

# # ------------------------------------------------------------
# # Variables
# # ------------------------------------------------------------
# delivery_dates = pd.date_range(product_parameters['delivery_start'], product_parameters['delivery_end'], freq='MS') # MS: month start frequency
# flist = [df.loc[df["file_date"] == delivery_dates[i].strftime("%Y%m"), "file_path"].iloc[0] for i in range(len(delivery_dates))]
# ds = xr.open_mfdataset(flist, combine='nested', concat_dim='time')

# # use earthkit to list only the fields of interest, excluding coordinate variables and such
# variables = list(ekd.from_source("file", flist[0]).to_fieldlist().ls()['variable'])

# styles = {
#     "precipitation": {
#         "cmap": "gist_earth_r",
#         "levels": [0,0.5,1,2,5,10,20,50,100],
#     },
#     "num_obs_fraction": {
#         "cmap": "Greens",
#         "levels": np.linspace(0,1,11),
#     },
#     "num_obs_rate": {
#         "cmap": "Purples",
#         "levels": np.linspace(0,1,11),
#     },
#     "num_days": {
#         "cmap": "Blues",
#         "levels": np.arange(0,32,2),
#     },
#     "quality_flag": {
#         "cmap": "tab10",
#         "levels": np.arange(-0.5,6.5,1),
#     },
#     "num_days_snow": {
#         "cmap": "cool",
#         "levels": np.arange(0,33,2),
#     },
# }

# # ------------------------------------------------------------
# # Dropdown
# # ------------------------------------------------------------

# variable_widget = widgets.Dropdown(
#     options=variables,
#     value="precipitation",
#     description="Variable:",
#     layout=widgets.Layout(width="350px")
# )

# output = widgets.Output()

# # ------------------------------------------------------------
# # Animation
# # ------------------------------------------------------------
# # import matplotlib as mpl

# # mpl.rcParams["animation.embed_limit"] = 100  # MB

# print(
#     "Animation embed limit:",
#     mpl.rcParams["animation.embed_limit"],
#     "MB"
# )
# def make_animation(variable):

#     style = styles[variable]

#     data = ds[variable]

#     fig = plt.figure(figsize=(10, 5), dpi=80)
#     ax = plt.axes(projection=ccrs.PlateCarree())

#     norm = mcolors.BoundaryNorm(
#         style["levels"],
#         ncolors=plt.get_cmap(style["cmap"]).N,
#         clip=True,
#     )

#     image = ax.imshow(
#         data.isel(time=0).values,
#         origin="lower",
#         extent=[
#             float(ds.lon.min()),
#             float(ds.lon.max()),
#             float(ds.lat.min()),
#             float(ds.lat.max()),
#         ],
#         transform=ccrs.PlateCarree(),
#         cmap=style["cmap"],
#         norm=norm,
#         interpolation="nearest",
#     )

#     cbar = plt.colorbar(
#         image,
#         ax=ax,
#         shrink=0.8,
#         pad=0.03,
#     )

#     units = data.attrs.get("units", "")

#     if units:
#         cbar.set_label(units)

#     ax.coastlines()
#     ax.add_feature(
#         cfeature.BORDERS,
#         linewidth=0.5
#     )

#     gl = ax.gridlines(
#         draw_labels=True,
#         linewidth=0.5,
#         linestyle="--",
#     )

#     gl.top_labels = False
#     gl.right_labels = False

#     title = ax.set_title("")

#     def update(frame):

#         image.set_data(
#             data.isel(time=frame).values
#         )

#         title.set_text(
#             f"{variable.replace('_', ' ')}\n"
#             f"{np.datetime_as_string(ds.time.values[frame], unit='M')}"
#         )

#         return image, title

#     ani = FuncAnimation(
#         fig,
#         update,
#         frames=data.sizes["time"],
#         interval=300,
#         blit=False,
#     )

#     plt.close(fig)

#     return HTML(ani.to_jshtml())

# # ------------------------------------------------------------
# # Callback
# # ------------------------------------------------------------

# def refresh(change=None):

#     with output:

#         output.clear_output(wait=True)

#         try:
#             display(
#                 make_animation(
#                     variable_widget.value
#                 )
#             )

#         except Exception as e:
#             print(
#                 f"Error while creating animation for "
#                 f"'{variable_widget.value}':"
#             )
#             print(e)
#             raise


# variable_widget.observe(refresh, names="value")

# display(variable_widget)
# display(output)

# refresh()

# # print(variables)
# # print(set(variables) - set(styles.keys()))

# ============================================================
# GIRAFE interactive map viewer
# Lightweight Voilà version
# ============================================================

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

import cartopy.crs as ccrs
import cartopy.feature as cfeature

import ipywidgets as widgets
from IPython.display import display


# ============================================================
# Prepare file list
# ============================================================

delivery_dates = pd.date_range(
    product_parameters["delivery_start"],
    product_parameters["delivery_end"],
    freq="MS"
)

flist = [
    df.loc[
        df["file_date"] == date.strftime("%Y%m"),
        "file_path"
    ].iloc[0]
    for date in delivery_dates
]


# ============================================================
# Plotting styles
# ============================================================

styles = {

    "precipitation": {
        "cmap": "gist_earth_r",
        "levels": [0, 0.5, 1, 2, 5, 10, 20, 50, 100],
    },

    "num_obs_fraction": {
        "cmap": "Greens",
        "levels": np.linspace(0, 1, 11),
    },

    "num_obs_rate": {
        "cmap": "Purples",
        "levels": np.linspace(0, 1, 11),
    },

    "num_days": {
        "cmap": "Blues",
        "levels": np.arange(0, 32, 2),
    },

    "quality_flag": {
        "cmap": "tab10",
        "levels": np.arange(-0.5, 6.5, 1),
    },

    "num_days_snow": {
        "cmap": "cool",
        "levels": np.arange(0, 33, 2),
    },
}


# ============================================================
# Variables
# ============================================================

# We already know which variables we want to display,
# so there is no need to inspect the first file with Earthkit.

variables = list(styles.keys())


# ============================================================
# Number of time steps
# ============================================================

n_times = len(flist)


# ============================================================
# Variable selector
# ============================================================

variable_widget = widgets.Dropdown(
    options=variables,
    value="precipitation",
    description="Variable:",
    layout=widgets.Layout(width="350px")
)


# ============================================================
# Time slider
# ============================================================

time_widget = widgets.IntSlider(
    value=0,
    min=0,
    max=n_times - 1,
    step=1,
    description="Time:",
    continuous_update=False,
    layout=widgets.Layout(width="650px")
)


# ============================================================
# Play button
# ============================================================

play_widget = widgets.Play(
    value=0,
    min=0,
    max=n_times - 1,
    step=1,
    interval=500,
    description="Play"
)


# Link Play button to slider
widgets.jslink(
    (play_widget, "value"),
    (time_widget, "value")
)


# ============================================================
# Date label
# ============================================================

date_label = widgets.Label(
    value=delivery_dates[0].strftime("%B %Y")
)


# ============================================================
# Output area
# ============================================================

map_output = widgets.Output()


# ============================================================
# Show a single map
# ============================================================

def show_map(variable, frame):

    style = styles[variable]

    # --------------------------------------------------------
    # Open ONLY the selected file
    # --------------------------------------------------------

    with xr.open_dataset(flist[frame]) as ds_month:

        data = ds_month[variable]

        # Monthly files contain one time step
        if "time" in data.dims:
            data = data.isel(time=0)

        # ----------------------------------------------------
        # Create figure
        # ----------------------------------------------------

        fig = plt.figure(
            figsize=(10, 5),
            dpi=80
        )

        ax = plt.axes(
            projection=ccrs.PlateCarree()
        )

        # ----------------------------------------------------
        # Colour scale
        # ----------------------------------------------------

        cmap = plt.get_cmap(style["cmap"])

        norm = mcolors.BoundaryNorm(
            style["levels"],
            ncolors=cmap.N,
            clip=True
        )

        # ----------------------------------------------------
        # Map
        # ----------------------------------------------------

        image = ax.imshow(
            data.values,
            origin="lower",
            extent=[
                float(ds_month.lon.min()),
                float(ds_month.lon.max()),
                float(ds_month.lat.min()),
                float(ds_month.lat.max())
            ],
            transform=ccrs.PlateCarree(),
            cmap=cmap,
            norm=norm,
            interpolation="nearest"
        )

        # ----------------------------------------------------
        # Colourbar
        # ----------------------------------------------------

        cbar = plt.colorbar(
            image,
            ax=ax,
            shrink=0.8,
            pad=0.03
        )

        units = data.attrs.get("units", "")

        if units:
            cbar.set_label(units)

        # ----------------------------------------------------
        # Geographic features
        # ----------------------------------------------------

        ax.coastlines()

        ax.add_feature(
            cfeature.BORDERS,
            linewidth=0.5
        )

        # ----------------------------------------------------
        # Gridlines
        # ----------------------------------------------------

        gl = ax.gridlines(
            draw_labels=True,
            linewidth=0.5,
            linestyle="--"
        )

        gl.top_labels = False
        gl.right_labels = False

        # ----------------------------------------------------
        # Title
        # ----------------------------------------------------

        date = delivery_dates[frame]

        ax.set_title(
            f"{variable.replace('_', ' ').title()}\n"
            f"{date.strftime('%B %Y')}"
        )

        # ----------------------------------------------------
        # Display
        # ----------------------------------------------------

        plt.show()

        # Close the figure so figures don't accumulate
        plt.close(fig)


# ============================================================
# Update map
# ============================================================

def update_map(change=None):

    frame = time_widget.value

    # Update date label
    date_label.value = delivery_dates[frame].strftime(
        "%B %Y"
    )

    # Update map
    with map_output:

        map_output.clear_output(
            wait=True
        )

        try:

            show_map(
                variable_widget.value,
                frame
            )

        except Exception as e:

            print(
                f"Error displaying "
                f"{variable_widget.value} "
                f"for {delivery_dates[frame].strftime('%Y-%m')}:"
            )

            print(e)


# ============================================================
# Connect widgets
# ============================================================

variable_widget.observe(
    update_map,
    names="value"
)

time_widget.observe(
    update_map,
    names="value"
)


# ============================================================
# Dashboard layout
# ============================================================

controls = widgets.VBox([

    variable_widget,

    widgets.HBox([
        play_widget,
        time_widget,
        date_label
    ])
])


display(
    widgets.VBox([
        controls,
        map_output
    ])
)


# ============================================================
# Display initial map
# ============================================================

# # ============================================================
# # Display initial map without triggering the callback
# # ============================================================

# with map_output:
#     show_map(
#         variable_widget.value,
#         time_widget.value
#     )

NameError: name 'df' is not defined

In [ ]:

# The final gallery builder will include the existing pre-calculated
# GIRAFE_monthly_stats.nc and, when available, GIRAFE_daily_stats.nc
# as interactive Plotly QC time-series panels.


In [ ]:

# The final gallery builder creates the static Cartopy PNG gallery under:
# GIRAFE_QC_gallery/maps/<frequency>/<variable>/


In [ ]:
# Generate the complete static GIRAFE QC viewer.
%run build_GIRAFE_QC_gallery.py


The global mean shows a jump in late 2023 that could be associated to the strong El Niño, since the same jump is not visible in the individual locations above.


### Publish to ECMWF Sites and embed in Confluence

The final cell creates `GIRAFE_QC_timeseries.html` and the accompanying `GIRAFE_QC_gallery/maps/` directory. Publish the whole `GIRAFE_QC_gallery` directory to ECMWF Sites without changing the relative paths.

Then embed the published HTML in Confluence, for example:

```html
<iframe src="https://sites.ecmwf.int/<your-site>/GIRAFE_QC_gallery/GIRAFE_QC_timeseries.html" width="100%" height="1800" frameborder="0"></iframe>
```
